In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os 
import random


In [ ]:
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Data Preprocessing

## Dataset split into train and validation split (80/20)

In [ ]:
import shutil
import random
import os

# Paths
src_dir = "/kaggle/input/datasets/utsavratapiya/person-identification-dataset/dataset"
base_dir = "/kaggle/working/split_dataset/"

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

# Create folders
for split_dir in [train_dir, val_dir]:
    for person in os.listdir(src_dir):
        os.makedirs(os.path.join(split_dir, person), exist_ok=True)

# Split EXACT 800-200
for person in os.listdir(src_dir):
    person_path = os.path.join(src_dir, person)
    images = os.listdir(person_path)
    
    # Safety check
    if len(images) < 1000:
        print(f"⚠️ Warning: {person} has less than 1000 images")
    
    random.shuffle(images)
    
    train_imgs = images[:800]
    val_imgs = images[800:1000]

    # Copy train
    for img in train_imgs:
        shutil.copy(
            os.path.join(person_path, img),
            os.path.join(train_dir, person, img)
        )

    # Copy validation
    for img in val_imgs:
        shutil.copy(
            os.path.join(person_path, img),
            os.path.join(val_dir, person, img)
        )

print("Perfect 80-20 split done! (800 train / 200 val per class)")

In [ ]:
# Original image size (camera / dataset reference)
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# Model input size (CNN requirement)
IMG_WIDTH = 224
IMG_HEIGHT = 224

# Training parameters
BATCH_SIZE = 32
EPOCHS = 15
SEED = 42


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Train generator (with augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

# Validation (NO augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

# Train generator
train_generator = train_datagen.flow_from_directory(
    "/kaggle/working/split_dataset/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

# Validation generator
val_generator = val_datagen.flow_from_directory(
    "/kaggle/working/split_dataset/val",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:

import matplotlib.pyplot as plt

base_dir = "/kaggle/working/split_dataset"

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

# Count images
def count_images(directory):
    total = 0
    for cls in os.listdir(directory):
        total += len(os.listdir(os.path.join(directory, cls)))
    return total

train_total = count_images(train_dir)
val_total = count_images(val_dir)

labels = ['Train', 'Validation']
sizes = [train_total, val_total]

plt.figure(figsize=(6,6))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
plt.title("Dataset Split Ratio (80-20)")
plt.show()

In [ ]:
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

In [ ]:

base_dir = "/kaggle/working/split_dataset"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

classes = sorted(os.listdir(train_dir))

train_counts = []
val_counts = []

# Count images per class
for cls in classes:
    train_count = len(os.listdir(os.path.join(train_dir, cls)))
    val_count = len(os.listdir(os.path.join(val_dir, cls)))
    
    train_counts.append(train_count)
    val_counts.append(val_count)

# Bar positions
x = np.arange(len(classes))
width = 0.35

plt.figure(figsize=(12,6))

# Bars
plt.bar(x - width/2, train_counts, width, label='Train')
plt.bar(x + width/2, val_counts, width, label='Validation')

# Labels
plt.xlabel("Person (Class)")
plt.ylabel("Number of Images")
plt.title("Class-wise Train vs Validation Split")
plt.xticks(x, classes, rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

## Random sample of training images with corresponding class labels

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(train_generator)

plt.figure(figsize=(10, 10))

for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    
    plt.imshow(images[i])
    class_index = np.argmax(labels[i])
    plt.title(class_names[class_index])
    
    plt.axis("off")

plt.show()

# CNN

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    data_augmentation,

    # Block 1
    layers.Conv2D(32, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(32, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(64, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Conv2D(128, (3,3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),

    # Block 4 (NEW )
    ## layers.Conv2D(256, (3,3), padding='same'),
    ## layers.BatchNormalization(),
    ## layers.Activation('relu'),
    ## layers.MaxPooling2D(),
    ## layers.Dropout(0.3),

    layers.GlobalAveragePooling2D(),  # better than Flatten

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)


In [ ]:

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

## Validation Results

In [ ]:
best_val_acc = max(history.history['val_accuracy'])
best_val_loss = min(history.history['val_loss'])

print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Best Validation Loss: {best_val_loss:.4f}")

In [ ]:
# Load best saved model
model = tf.keras.models.load_model('best_model.keras')

val_loss, val_accuracy = model.evaluate(val_generator)

print(f"Final Validation Accuracy (Best Model): {val_accuracy:.4f}")
print(f"Final Validation Loss (Best Model): {val_loss:.4f}")

## Plots : Accuracy & loss vs Epoches

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

# Find best epoch
best_epoch = np.argmax(val_acc) + 1
best_val_acc = max(val_acc)

plt.figure(figsize=(14, 5))

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, marker='o', label='Training Accuracy')
plt.plot(epochs_range, val_acc, marker='o', label='Validation Accuracy')

# Mark best point
plt.scatter(best_epoch, best_val_acc, label=f'Best Epoch ({best_epoch})', s=100)

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.title('Training vs Validation Accuracy')

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, marker='o', label='Training Loss')
plt.plot(epochs_range, val_loss, marker='o', label='Validation Loss')

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.title('Training vs Validation Loss')

plt.tight_layout()
plt.show()

print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

In [ ]:
model.save("person_classification_cnn.h5")
print("Model saved successfully.")

## Manually Testing with Path

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load best model
model = load_model("best_model.keras")

def predict_person(image_path):
    img = cv2.imread(image_path)
    
    if img is None:
        raise ValueError("Image not found or path is incorrect!")
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    predictions = model.predict(img)
    predicted_index = np.argmax(predictions)
    confidence = np.max(predictions)

    return class_names[predicted_index], confidence


# Test image
test_image_path = "/kaggle/input/datasets/utsavratapiya/person-identification-dataset/dataset/Utsav/Utsav_119.jpg"

person, confidence = predict_person(test_image_path)

print("Predicted Person:", person)
print(f"Confidence: {confidence:.2f}")

## Testing random image from Validation data

In [ ]:
import os
import pandas as pd

base_dir = "/kaggle/working/split_dataset"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

classes = sorted(os.listdir(train_dir))

data = []

total_images = 0
total_train = 0
total_val = 0

for cls in classes:
    train_count = len(os.listdir(os.path.join(train_dir, cls)))
    val_count = len(os.listdir(os.path.join(val_dir, cls)))
    total_count = train_count + val_count

    total_images += total_count
    total_train += train_count
    total_val += val_count

    data.append([cls, total_count, train_count, val_count])

# Create DataFrame
df = pd.DataFrame(data, columns=["Class", "Overall", "Train", "Validation"])

# Add Total Row
df.loc[len(df)] = ["Total", total_images, total_train, total_val]

df

In [ ]:


# Dataset path (original dataset, not split)
dataset_path = "/kaggle/working/split_dataset/val"

# Step 1: Pick random class
random_class = random.choice(os.listdir(dataset_path))
class_path = os.path.join(dataset_path, random_class)

# Step 2: Pick random image from that class
random_image = random.choice(os.listdir(class_path))
image_path = os.path.join(class_path, random_image)

print("Selected Image:", image_path)

# Step 3: Read & preprocess
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (IMG_WIDTH, IMG_HEIGHT))
img_normalized = img_resized / 255.0
img_exp = np.expand_dims(img_normalized, axis=0)

# Step 4: Prediction
predictions = model.predict(img_exp)
predicted_index = np.argmax(predictions)
confidence = np.max(predictions)

predicted_person = class_names[predicted_index]

# Step 5: Show image + result
plt.imshow(img_rgb)
plt.title(f"Predicted: {predicted_person}\nConfidence: {confidence:.2f}")
plt.axis("off")
plt.show()

## Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Get true labels
y_true = val_generator.classes

# Predict all test data
y_pred_probs = model.predict(val_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

# Class names
class_labels = list(val_generator.class_indices.keys())

# Classification Report
report = classification_report(y_true, y_pred, target_names=class_labels)

print("Classification Report:\n")
print(report)

## Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels,
            yticklabels=class_labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


## Final Predection using CNN

In [ ]:

dataset_path = "/kaggle/input/datasets/utsavratapiya/person-identification-dataset/dataset"

plt.figure(figsize=(15, 10))

for i in range(10):
    # Random class
    random_class = random.choice(os.listdir(dataset_path))
    class_path = os.path.join(dataset_path, random_class)

    # Random image from that class
    img_name = random.choice(os.listdir(class_path))
    img_path = os.path.join(class_path, img_name)

    # Read image
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess
    img_resized = cv2.resize(img_rgb, (224, 224)) / 255.0
    img_exp = np.expand_dims(img_resized, axis=0)

    # Prediction
    pred = model.predict(img_exp)
    pred_class = np.argmax(pred)
    confidence = np.max(pred)

    # Plot
    plt.subplot(2, 5, i+1)
    plt.imshow(img_rgb)
    plt.title(f"Pred: {class_names[pred_class]}\nTrue: {random_class}\nConf: {confidence:.2f}")
    plt.axis('off')

plt.tight_layout()
plt.show()

# MobileNet 

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model_mobilenet = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_mobilenet = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=12,
    callbacks=[early_stop,reduce_lr]
)

In [ ]:
mobilenet_loss, mobilenet_acc = model_mobilenet.evaluate(val_generator, verbose=0)

print(f" MobileNetV2 Validation Accuracy: {mobilenet_acc:.4f}")
print(f" MobileNetV2 Validation Loss: {mobilenet_loss:.4f}")

In [ ]:
model_mobilenet.save("mobilenet_model.h5")

# VGG-16

In [ ]:
from tensorflow.keras.applications import VGG16

base_model = VGG16(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model_vgg = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model_vgg.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_vgg = model_vgg.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
vgg_loss, vgg_acc = model_vgg.evaluate(val_generator, verbose=0)

print(f" VGG16 Validation Accuracy: {vgg_acc:.4f}")
print(f" VGG16 Validation Loss: {vgg_loss:.4f}")

In [ ]:
model_vgg.save("VGG16_model.h5")

# Efficient net

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model_efficient = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),   # better than Flatten for EfficientNet

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model_efficient.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',   # matches your generator
    metrics=['accuracy']
)

history_efficient = model_efficient.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
val_loss, val_acc = model_efficient.evaluate(val_generator)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

In [ ]:
model_efficient.save("efficientnet_fast.h5")

# Model Comparision

In [ ]:
cnn_acc = 0.9157
mobilenet_acc = 1.00
eff_acc = 0.9786   # close to MobileNet
vgg_acc = 1.00   

models = ["CNN", "MobileNetV2", "EfficientNetB0", "VGG16"]
accuracies = [cnn_acc, mobilenet_acc, eff_acc, vgg_acc]

plt.figure(figsize=(8,5))
bars = plt.bar(models, accuracies)

plt.xlabel("Models")
plt.ylabel("Validation Accuracy")
plt.title("Model Accuracy Comparison")

# Show values on bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01,
             f"{yval:.3f}", ha='center', fontsize=10)

plt.ylim(0, 1.1)
plt.grid(False)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extract accuracies
cnn_acc = history.history['val_accuracy']
mobilenet_acc = history_mobilenet.history['val_accuracy']
vgg_acc = history_vgg.history['val_accuracy']

# Epoch ranges
epochs_cnn = range(1, len(cnn_acc) + 1)
epochs_mob = range(1, len(mobilenet_acc) + 1)
epochs_vgg = range(1, len(vgg_acc) + 1)

plt.figure(figsize=(10,6))

# Plot curves
plt.plot(epochs_cnn, cnn_acc, marker='o', label='CNN')
plt.plot(epochs_mob, mobilenet_acc, marker='o', label='MobileNetV2')
plt.plot(epochs_vgg, vgg_acc, marker='o', label='VGG16')

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Model Comparison Curve")

plt.legend()
plt.grid(True)

plt.show()